In [1]:
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd


In [2]:
df=pd.read_csv(r"C:\Users\shahh\OneDrive\Desktop\Data Science And Machine Learning\practice\SmartPhonePrediction\train.csv")

In [3]:
df.drop(columns=["id"],inplace=True)

In [4]:
numerical_cols = [
    'daily_screen_time_hours',
    'social_media_hours',
    'gaming_hours',
    'work_study_hours',
    'sleep_hours',
    'notifications_per_day',
    'app_opens_per_day',
    'weekend_screen_time',
    'age'
]


In [5]:
df.dropna(thresh=8,inplace=True)

In [6]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler





scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[numerical_cols])

imputer = IterativeImputer(
    max_iter=10,
    random_state=42
)

imputed_data = imputer.fit_transform(scaled_data)

# Convert back to original scale
imputed_data = scaler.inverse_transform(imputed_data)

# Put values back into dataframe
df[numerical_cols] = imputed_data

c:\Users\shahh\OneDrive\Desktop\Data Science And Machine Learning\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [7]:
#Feature Engineering
df["entertainment_hours"] = (
    df["social_media_hours"] +
    df["gaming_hours"]
)


df["entertainment_ratio"] = (
    df["entertainment_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["social_media_ratio"] = (
    df["social_media_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["gaming_ratio"] = (
    df["gaming_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["work_study_ratio"] = (
    df["work_study_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["weekend_screen_ratio"] = (
    df["weekend_screen_time"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["weekend_screen_difference"] = (
    df["weekend_screen_time"] -
    df["daily_screen_time_hours"]
)


df["recreational_to_work_ratio"] = (
    df["entertainment_hours"] /
    (df["work_study_hours"] + 0.01)
)


df["social_gaming_interaction"] = (
    df["social_media_hours"] *
    df["gaming_hours"]
)


df["sleep_screen_ratio"] = (
    df["sleep_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)

In [8]:
categorical_cols = [
    "gender",
    "stress_level",
    "academic_work_impact"
    
]

In [9]:
df[categorical_cols] = df[categorical_cols].fillna("Unknown")

In [10]:
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [11]:
X=df.drop("addicted_label",axis=1)
y=df['addicted_label']

In [12]:
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42,stratify=y)

In [14]:
scaler=StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu126
CUDA available: True
CUDA version: 12.6
Device: cuda
GPU: NVIDIA GeForce MX350


In [16]:
import sys
print(sys.executable)


c:\Users\shahh\OneDrive\Desktop\Data Science And Machine Learning\.venv\Scripts\python.exe


In [17]:
# X and y are pandas/NumPy objects at this stage.
# They are converted to PyTorch tensors after scaling.


In [18]:
# The model is moved to the selected device after it is created.


In [19]:
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
).to(device)

print(model)


Sequential(
  (0): Linear(in_features=27, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
)


In [20]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


In [21]:
# Convert the already-scaled data to PyTorch tensors
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32,
    device=device
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32,
    device=device
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32,
    device=device
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32,
    device=device
).reshape(-1, 1)

# Train the ANN
epochs = 100

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            predictions = (torch.sigmoid(outputs) >= 0.5).float()
            accuracy = (predictions == y_train_tensor).float().mean()

        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Loss: {loss.item():.4f} "
            f"Accuracy: {accuracy.item():.4f}"
        )

# Test the model
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_predictions = (
        torch.sigmoid(test_outputs) >= 0.5
    ).float()

    test_accuracy = (
        test_predictions == y_test_tensor
    ).float().mean()

print(f"Test Accuracy: {test_accuracy.item():.4f}")


Epoch [10/100] Loss: 0.7167 Accuracy: 0.3513
Epoch [20/100] Loss: 0.6920 Accuracy: 0.4733
Epoch [30/100] Loss: 0.6625 Accuracy: 0.6153
Epoch [40/100] Loss: 0.6275 Accuracy: 0.7164
Epoch [50/100] Loss: 0.5876 Accuracy: 0.7632
Epoch [60/100] Loss: 0.5443 Accuracy: 0.7831
Epoch [70/100] Loss: 0.5009 Accuracy: 0.7954
Epoch [80/100] Loss: 0.4611 Accuracy: 0.8074
Epoch [90/100] Loss: 0.4272 Accuracy: 0.8191
Epoch [100/100] Loss: 0.3998 Accuracy: 0.8296
Test Accuracy: 0.8304


In [22]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.0+cu126
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce MX350
